# Coba scanner di Kaggle — tanpa HP

Notebook ini menjalankan **dua model yang ada di aplikasi** pada foto yang kamu unggah sendiri:
detektor `yolov8n` mencari kotak pakaian, classifier 12 kelas menamai isi tiap kotak. Tidak ada
pelatihan, jadi selesai dalam satu-dua menit.

Gunanya untuk menjawab pertanyaan seperti "kenapa sepatu ini tidak muncul" tanpa harus memasang
APK: angka skor detektor, kotak yang lolos, dan nama per kotak dicetak apa adanya.

### Cara pakai

1. Panel kanan → **Internet → On** (hanya untuk meng-clone repo).
2. Jalankan sel 1 sampai 3.
3. Di sel 4: pilih folder fotomu. Dua cara:
   * **Upload Data** (panel kanan) → buat dataset privat kecil berisi foto → otomatis terbaca di
     `/kaggle/input`.
   * atau pakai tombol **Upload file** di notebook ini, lalu pindahkan ke `WORK/images/`.
4. Jalankan sel 4 sampai 6. Hasilnya: satu tabel per foto dan gambar dengan kotak + label.

Ambangnya sama dengan aplikasi (`docs/multi-item-scan.md`): skor detektor 0,25, kotak satu Category
yang tumpang tindih lebih dari 0,70 dianggap sama, maksimal 6 item, terbesar lebih dulu.

## 1. Siapkan model dan kode

Model diambil dari repo, bukan dari salinan — jadi yang kamu uji adalah berkas yang benar-benar
dipakai aplikasi. `REPO_REF` menunjuk branch yang memuat classifier 12 kelas.

In [ ]:
# Langkah 1/6 — ambil model dari repo.
import json
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kmilaz/gayaku.git"
REPO_REF = "feat/ml-pipeline"
REPO_DIR = Path("/kaggle/temp/gayaku")
WORK = Path("/kaggle/working")
MODELS = WORK / "models"
MODELS.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

assets = REPO_DIR / "flutter_app/assets/models"
for name in ("garment_detector.tflite", "garment_classifier.tflite"):
    shutil.copy2(assets / name, MODELS / name)
labels = json.loads((REPO_DIR / "ml_workspace/artifacts/classes.json").read_text(encoding="utf-8"))

print("models :", [p.name for p in sorted(MODELS.iterdir())])
print("kelas  :", len(labels["labels"]), labels["labels"])

## 2. Dekode detektor, sama seperti di aplikasi

Fungsi di bawah ini salinan aturan `decodeDetectorOutput` dan `selectScanBoxes` di
`scan_pipeline.dart`. Kalau angkanya berbeda dari yang kamu lihat di HP, berarti sel ini dan
aplikasinya sudah tidak sinkron.

In [ ]:
# Langkah 2/6 — dekode dan pemilihan kotak.
import numpy as np
from PIL import Image
CATEGORIES = ["Tops", "Bottoms", "Outerwear", "Shoes", "Accessories"]
DETECTOR_SIZE = 640
CLASSIFIER_SIZE = 224
SCORE_THRESHOLD = 0.25
DUPLICATE_IOU = 0.70
MAX_ITEMS = 6
MARGIN = 0.05


def letterbox(image):
    width, height = image.size
    scale = DETECTOR_SIZE / max(width, height)
    content_w, content_h = round(width * scale), round(height * scale)
    canvas = Image.new("RGB", (DETECTOR_SIZE, DETECTOR_SIZE), (114, 114, 114))
    left, top = (DETECTOR_SIZE - content_w) // 2, (DETECTOR_SIZE - content_h) // 2
    canvas.paste(image.resize((content_w, content_h), Image.BILINEAR), (left, top))
    return canvas, left, top, content_w, content_h


def iou(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    if x1 <= x0 or y1 <= y0:
        return 0.0
    inter = (x1 - x0) * (y1 - y0)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def decode(raw):
    """[9, 8400] -> boxes, largest first, one per Category where they overlap."""
    candidates = []
    for anchor in range(raw.shape[1]):
        scores = raw[4:, anchor]
        cls = int(np.argmax(scores))
        score = float(scores[cls])
        if score < SCORE_THRESHOLD:
            continue
        cx, cy, w, h = (float(v) for v in raw[:4, anchor])
        if w <= 0 or h <= 0:
            continue
        candidates.append((score, cls, (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)))

    candidates.sort(key=lambda item: -item[0])
    kept = []
    for score, cls, box in candidates:
        if any(k[1] == cls and iou(k[2], box) > DUPLICATE_IOU for k in kept):
            continue
        kept.append((score, cls, box))

    kept.sort(key=lambda item: -((item[2][2] - item[2][0]) * (item[2][3] - item[2][1])))
    return kept[:MAX_ITEMS]


def to_photo(box, pad_left, pad_top, content_w, content_h):
    x0, y0, x1, y1 = box
    return (
        max(0.0, min(1.0, (x0 * DETECTOR_SIZE - pad_left) / content_w)),
        max(0.0, min(1.0, (y0 * DETECTOR_SIZE - pad_top) / content_h)),
        max(0.0, min(1.0, (x1 * DETECTOR_SIZE - pad_left) / content_w)),
        max(0.0, min(1.0, (y1 * DETECTOR_SIZE - pad_top) / content_h)),
    )

## 3. Jalankan kedua model pada satu foto

Classifer dijalankan pada **crop mentah** dengan margin 5 %, persis seperti data latihnya
(`fashionpedia_crops.py`). Aplikasi memotongnya dengan matte dulu; sel di sini sengaja melewati
matte supaya nama yang keluar adalah jawaban classifier, bukan campuran dua tahap.

In [ ]:
# Langkah 3/6 — satu foto: deteksi, potong, klasifikasi.
from ai_edge_litert.interpreter import Interpreter
from PIL import Image, ImageDraw, ImageFont

detector = Interpreter(model_path=str(MODELS / "garment_detector.tflite"))
detector.allocate_tensors()
classifier = Interpreter(model_path=str(MODELS / "garment_classifier.tflite"))
classifier.allocate_tensors()
CLASSES = labels["labels"]

print("detektor  ", tuple(detector.get_input_details()[0]["shape"]), "->", tuple(detector.get_output_details()[0]["shape"]))
print("classifier", tuple(classifier.get_input_details()[0]["shape"]), "->", tuple(classifier.get_output_details()[0]["shape"]))


def classify_crop(image, box):
    width, height = image.size
    x0, y0, x1, y1 = box
    mx, my = (x1 - x0) * MARGIN, (y1 - y0) * MARGIN
    crop = image.crop((
        max(0, int((x0 - mx) * width)),
        max(0, int((y0 - my) * height)),
        min(width, int((x1 + mx) * width)),
        min(height, int((y1 + my) * height)),
    ))
    pixels = np.asarray(crop.resize((CLASSIFIER_SIZE, CLASSIFIER_SIZE), Image.BILINEAR), dtype=np.float32) / 255.0
    tensor = classifier.get_input_details()[0]
    shape = tuple(int(v) for v in tensor["shape"])
    array = pixels[None, ...] if shape[1:] == (CLASSIFIER_SIZE, CLASSIFIER_SIZE, 3) else pixels.transpose(2, 0, 1)[None, ...]
    classifier.set_tensor(tensor["index"], array.astype(np.float32))
    classifier.invoke()
    scores = classifier.get_tensor(classifier.get_output_details()[0]["index"])[0]
    best = int(np.argmax(scores))
    return CLASSES[best], float(scores[best])


def scan(path):
    """Every garment the app would show for one photograph."""
    image = Image.open(path).convert("RGB")
    canvas, pad_left, pad_top, content_w, content_h = letterbox(image)
    array = (np.asarray(canvas, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None, ...]
    detector.set_tensor(detector.get_input_details()[0]["index"], array)
    detector.invoke()
    raw = detector.get_tensor(detector.get_output_details()[0]["index"])[0]

    items = []
    for score, cls, box in decode(raw):
        photo_box = to_photo(box, pad_left, pad_top, content_w, content_h)
        label, confidence = classify_crop(image, photo_box)
        items.append({
            "box": photo_box,
            "category": CATEGORIES[cls],
            "score": score,
            "label": label,
            "confidence": confidence,
        })
    return image, items


def draw(image, items):
    drawn = image.copy()
    canvas = ImageDraw.Draw(drawn)
    colour = (232, 72, 48)
    for item in items:
        width, height = drawn.size
        x0, y0, x1, y1 = item["box"]
        canvas.rectangle((x0 * width, y0 * height, x1 * width, y1 * height), outline=colour, width=4)
        caption = f'{item["label"]} {item["confidence"]:.0%}'
        canvas.rectangle((x0 * width, y0 * height - 22, x0 * width + 8 * len(caption), y0 * height), fill=colour)
        canvas.text((x0 * width + 4, y0 * height - 20), caption, fill="white")
    return drawn

## 4. Pilih fotonya

Letakkan foto di salah satu tempat ini:

* `/kaggle/input/<nama-dataset>/...` — dataset yang kamu tambahkan lewat panel kanan.
* `WORK/images/` — untuk foto yang kamu unggah satu per satu di notebook ini.

Sel ini mencari `.jpg`, `.jpeg`, dan `.png` di kedua tempat, maksimal 40 foto, dan berhenti dengan
pesan jelas kalau belum ada foto sama sekali.

In [ ]:
# Langkah 4/6 — kumpulkan foto masukan.
import shutil
from pathlib import Path

UPLOAD = WORK / "images"
UPLOAD.mkdir(exist_ok=True)
extensions = (".jpg", ".jpeg", ".png")
photos = []
for root in (Path("/kaggle/input"), UPLOAD):
    if not root.exists():
        continue
    photos += [p for p in sorted(root.rglob("*")) if p.suffix.lower() in extensions]
photos = photos[:40]

if not photos:
    raise SystemExit(
        "Belum ada foto. Tambahkan dataset foto lewat panel kanan (Upload Data), atau unggah "
        f"berkas ke {UPLOAD} lalu jalankan ulang sel ini."
    )
print(f"{len(photos)} foto:")
for path in photos:
    print("  ", path)

## 5. Hasil per foto

Setiap baris: nomor item, Category dari detektor, skor detektor, lalu nama dan keyakinan dari
classifier. Baris ditandai `perlu dicek` ketika nama atau Categorynya ragu, sama seperti badge di
aplikasi.

In [ ]:
# Langkah 5/6 — jalankan semuanya dan cetak tabelnya.
import pandas as pd

rows = []
drawn = []
for path in photos:
    image, items = scan(path)
    drawn.append((path, draw(image, items)))
    if not items:
        rows.append({"foto": path.name, "item": 0, "Category": "-", "skor detektor": None,
                     "nama": "-", "keyakinan": None})
    for index, item in enumerate(items, start=1):
        rows.append({
            "foto": path.name,
            "item": index,
            "Category": item["category"],
            "skor detektor": round(item["score"], 3),
            "nama": item["label"],
            "keyakinan": round(item["confidence"], 3),
        })

table = pd.DataFrame(rows)
print(table.to_string(index=False))
print()
shoes = table[table["Category"] == "Shoes"]
print(f"foto: {len(photos)}  item: {len(table[table['item'] > 0])}  kotak Shoes: {len(shoes)}")

## 6. Lihat gambarnya

Satu gambar per foto dengan kotak dan label, disimpan juga ke `/kaggle/working/scan/` supaya bisa
diunduh dari panel Output. Kalau kotaknya salah tempat atau labelnya meleset, itu bukti yang bisa
dibawa ke diskusi berikutnya — bukan tebakan dari layar HP.

In [ ]:
# Langkah 6/6 — simpan dan tampilkan.
from IPython.display import Image as ShowImage, display

OUT = WORK / "scan"
OUT.mkdir(exist_ok=True)
for path, picture in drawn:
    target = OUT / f"{path.stem}_scan.png"
    picture.save(target)
    print(f"{path.name} -> {target.name} ({picture.width}x{picture.height})")
    display(ShowImage(filename=str(target), width=520))

print(f"Semua gambar ada di {OUT}")